In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Get Device for Training

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


# Define the Class

In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [4]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [7]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([2], device='mps:0')


# Model Layers

In [13]:
input_image = torch.rand(3,28,28)
print(input_image.size())

torch.Size([3, 28, 28])


## nn.Flatten

In [15]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())

torch.Size([3, 784])


## nn.Linear

In [16]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())

torch.Size([3, 20])


## nn.ReLU

In [17]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.4810,  0.0923,  0.1265, -0.3351, -0.2921,  0.3886, -0.6405, -0.7943,
          0.3136,  0.0693,  0.1716, -0.4520,  0.5673, -0.3464,  0.0123, -0.4639,
         -0.3132,  0.2302, -0.1679, -0.2392],
        [ 0.5796,  0.2307,  0.2310, -0.1020,  0.0636,  0.4017, -0.4150, -0.4865,
          0.1601, -0.0333,  0.3606, -0.5184,  0.1567, -0.3239,  0.0129,  0.4424,
         -0.4592,  0.0987,  0.1884,  0.0560],
        [ 0.5687,  0.2112, -0.0400, -0.3625, -0.1144,  0.2827, -0.5621, -0.4740,
          0.4268,  0.0684,  0.6678, -0.3297, -0.0064,  0.0199,  0.0516, -0.1175,
         -0.4614,  0.0299, -0.0312, -0.0426]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.4810, 0.0923, 0.1265, 0.0000, 0.0000, 0.3886, 0.0000, 0.0000, 0.3136,
         0.0693, 0.1716, 0.0000, 0.5673, 0.0000, 0.0123, 0.0000, 0.0000, 0.2302,
         0.0000, 0.0000],
        [0.5796, 0.2307, 0.2310, 0.0000, 0.0636, 0.4017, 0.0000, 0.0000, 0.1601,
         0.0000, 0.3606, 0.0000, 0.1567, 0.0000, 0.01

## nn.Sequential

In [23]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3, 28, 28)
logits = seq_modules(input_image)
print(logits.shape)

torch.Size([3, 10])


## nn.Softmax

In [22]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

# Model Parameters

In [25]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values: {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values: tensor([[-0.0314, -0.0109, -0.0239,  ..., -0.0220,  0.0172,  0.0266],
        [-0.0110, -0.0135,  0.0040,  ...,  0.0271, -0.0040,  0.0021]],
       device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values: tensor([-0.0289,  0.0247], device='mps:0', grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values: tensor([[ 0.0086,  0.0324, -0.0308,  ...,  0.0199,  0.0347, -0.0193],
        [-0.0429, -0.0137,  0.0006,  ...,  0.0051, -0.0214, -0.0433]],
       device='mps:0', grad_fn=<SliceBac